In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import precision_recall_fscore_support
import warnings

# Import the dataset loader
from pytorch_dataset import EBSDStrainDataset

class EBSDStrainCNN(nn.Module):
    """
    CNN architecture adapted from the MNIST model to handle EBSD patterns.
    Uses AdaptiveAvgPool2d to accommodate different EBSD image resolutions.
    """
    def __init__(self):
        super(EBSDStrainCNN, self).__init__()

        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25)
        )

        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25)
        )

        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Dropout(0.25)
        )

        # Adaptive pooling ensures the flattened size is always 128 * 4 * 4
        # regardless of the original EBSD pattern dimensions
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)  # Output a single scalar for continuous strain prediction
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.adaptive_pool(x)
        x = self.fc_block(x)
        return x.squeeze()

def calculate_metrics(y_true, y_pred, strain_classes=None):
    """
    Calculates percent error, standard deviation, and classification metrics
    by snapping continuous predictions to the nearest known discrete strain classes.
    """
    epsilon = 1e-8
    percent_errors = np.abs((y_true - y_pred) / (y_true + epsilon)) * 100
    avg_percent_error = np.mean(percent_errors)
    std_dev_error = np.std(percent_errors)

    if strain_classes is None:
        # Ensure strain_classes are sorted for consistent mapping
        strain_classes = np.sort(np.unique(y_true))

    # Create a mapping from float strain values to integer labels for sklearn
    strain_to_int_map = {strain: i for i, strain in enumerate(strain_classes)}

    # Snap predictions and true values to the closest known strain classes (float values)
    y_pred_snapped_floats = np.array([strain_classes[np.argmin(np.abs(strain_classes - pred))] for pred in y_pred])
    y_true_snapped_floats = np.array([strain_classes[np.argmin(np.abs(strain_classes - true))] for true in y_true])

    # Convert the snapped float values to integer labels using the map
    y_pred_classes_int = np.array([strain_to_int_map[s] for s in y_pred_snapped_floats])
    y_true_classes_int = np.array([strain_to_int_map[s] for s in y_true_snapped_floats])

    warnings.filterwarnings('ignore')
    precision, recall, f1, _ = precision_recall_fscore_support(y_true_classes_int, y_pred_classes_int, average='weighted')

    return avg_percent_error, std_dev_error, precision, recall, f1

def train_and_evaluate(h5_path="ebsd_fcc_fe.h5", epochs=50, batch_size=32, learning_rate=0.0005):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    print(f"Loading data from: {h5_path}")

    # 1. Load Datasets
    train_dataset = EBSDStrainDataset(h5_path=h5_path, split="train")
    test_dataset = EBSDStrainDataset(h5_path=h5_path, split="test")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # 2. Initialize Model, Loss Function, and Optimizer
    model = EBSDStrainCNN().to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6)

    best_f1 = 0.0

    # 3. Training Loop
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, (patterns, strains, eulers) in enumerate(train_loader):
            patterns, strains = patterns.to(device), strains.to(device)

            optimizer.zero_grad()
            outputs = model(patterns)

            loss = criterion(outputs, strains)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        # 4. Evaluation Round
        model.eval()
        test_loss = 0.0
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for patterns, strains, eulers in test_loader:
                patterns, strains = patterns.to(device), strains.to(device)
                outputs = model(patterns)

                loss = criterion(outputs, strains)
                test_loss += loss.item()

                all_preds.extend(outputs.cpu().numpy())
                all_targets.extend(strains.cpu().numpy())

        test_loss /= len(test_loader)
        scheduler.step(test_loss)

        all_preds = np.array(all_preds)
        all_targets = np.array(all_targets)

        avg_pe, std_pe, precision, recall, f1 = calculate_metrics(all_targets, all_preds)

        print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {running_loss/len(train_loader):.4f} - Test Loss: {test_loss:.4f}")
        print(f"Metrics: P: {precision:.4f} | R: {recall:.4f} | F1: {f1:.4f} | % Error: {avg_pe:.2f}% (Std: {std_pe:.2f})")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "best_ebsd_strain_model.pth")

    print("\nTraining completed. Best model saved to 'best_ebsd_strain_model.pth'.")

if __name__ == "__main__":
    # The default file path is now directly integrated
    train_and_evaluate(h5_path="/content/drive/MyDrive/ebsd_fcc_fe.h5", epochs=50, batch_size=64)

Using device: cuda
Loading data from: /content/drive/MyDrive/ebsd_fcc_fe.h5
Epoch [1/50] - Train Loss: 470.6233 - Test Loss: 219.3633
Metrics: P: 0.4618 | R: 0.3350 | F1: 0.3530 | % Error: 61.80% (Std: 57.42)
Epoch [2/50] - Train Loss: 72.1018 - Test Loss: 29.0432
Metrics: P: 0.4864 | R: 0.4458 | F1: 0.4387 | % Error: 55.14% (Std: 93.93)
Epoch [3/50] - Train Loss: 42.7185 - Test Loss: 62.1165
Metrics: P: 0.3010 | R: 0.2717 | F1: 0.2777 | % Error: 64.23% (Std: 86.78)
Epoch [4/50] - Train Loss: 40.0222 - Test Loss: 35.3377
Metrics: P: 0.3681 | R: 0.3533 | F1: 0.3387 | % Error: 64.03% (Std: 96.49)
Epoch [5/50] - Train Loss: 37.0805 - Test Loss: 16.8766
Metrics: P: 0.5438 | R: 0.4992 | F1: 0.5002 | % Error: 33.92% (Std: 52.83)
Epoch [6/50] - Train Loss: 35.1780 - Test Loss: 20.9499
Metrics: P: 0.5278 | R: 0.5217 | F1: 0.4941 | % Error: 56.20% (Std: 90.52)
Epoch [7/50] - Train Loss: 33.8990 - Test Loss: 50.7548
Metrics: P: 0.3624 | R: 0.3917 | F1: 0.3451 | % Error: 92.22% (Std: 137.97)
Epoc